# Stage 8 — Retrieve Relevant Guidelines / Literature / ICD Codes

For each candidate diagnosis that survived pruning (Stage 7):
- **ICD-10 codes** — grounded search against MIMIC's real ICD-10-CM dictionary (`d_icd_diagnoses.csv.gz`,
  the same file used for ground truth in Stage 1), then the LLM picks the best match(es) from that real
  shortlist only. Any code the LLM returns that isn't in the shortlist is dropped as a defensive check.
- **Guideline relevance** — looked up in a curated static reference table (`GUIDELINE_REFERENCE_TAXONOMY`
  in `pipeline.py`), not invented by the LLM. The LLM writes a short note connecting that citation to the
  patient's specific evidence.

**Requires local MIMIC-IV access** (`d_icd_diagnoses.csv.gz`) — set `PHYSIONET_ROOT` in
`00_settings.ipynb` if this stage errors with a missing-file message.

**Note:** `GUIDELINE_REFERENCE_TAXONOMY` citations are curated best-effort references and should be
spot-checked by a clinician before any real use — a few diseases have no single dedicated society
guideline and are noted as such rather than assigned a citation that overstates precision.

**Input:** pruning per-patient files (Stage 7)
**Output:** one JSON file per patient in `data/stage_08_guideline_icd_retrieval/` + an index


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    LLMNotAvailableError,
    check_llm,
    get_llm_config,
    guideline_icd_retrieval_agent,
    load_icd10_dictionary,
    load_pruning_results,
    print_pipeline_banner,
    save_retrieval_results,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

icd_dict = load_icd10_dictionary()
print(f"ICD-10 dictionary loaded: {len(icd_dict)} codes")

pruning_df = load_pruning_results()
print(f"Admissions to retrieve for: {len(pruning_df)}")

## Retrieve ICD-10 Codes + Guideline Notes for Kept Candidates

In [ ]:
retrieved = []

for _, row in pruning_df.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    kept_candidates = row["branch_pruning"].get("kept_candidates") or []
    print(f"Retrieving patient={patient_id} hadm_id={hadm_id} ({len(kept_candidates)} kept candidates)...")
    result = guideline_icd_retrieval_agent(
        kept_candidates=kept_candidates,
        admission_id=hadm_id,
        patient_id=patient_id,
        config=LLM_CONFIG,
        icd_dict=icd_dict,
    )
    retrieved.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "guideline_icd_retrieval_method": result.get("_method", "unknown"),
        "guideline_icd_retrieval": result,
        "n_retrieved": len(result.get("retrieved") or []),
    })

results_df = pd.DataFrame(retrieved)
results_df[["patient_id", "admission_id", "guideline_icd_retrieval_method", "n_retrieved"]]

## Save Results

In [ ]:
out = save_retrieval_results(results_df)
print(f"Saved → {out}")